In [1]:
import pandas as pd
import gensim
import numpy as np
from gensim.models import Word2Vec
import io
import zipfile
import urllib.request

In [2]:
url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"

# Завантажуємо zip-файл у пам'ять
response = urllib.request.urlopen(url)
data = response.read()

# Відкриваємо архів
archive = zipfile.ZipFile(io.BytesIO(data))

# Читаємо дві потрібні таблиці
ratings = pd.read_csv(archive.open("ml-latest-small/ratings.csv"))
movies = pd.read_csv(archive.open("ml-latest-small/movies.csv"))

print("ratings:", ratings.shape)
print("movies:", movies.shape)

ratings.head()

ratings: (100836, 4)
movies: (9742, 3)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [3]:
positive_ratings = ratings[ratings["rating"] >= 4.0].copy()

print("Усі оцінки:", len(ratings))
print("Позитивні оцінки:", len(positive_ratings))

positive_ratings.head()

Усі оцінки: 100836
Позитивні оцінки: 48580


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [4]:
# Групуємо позитивно оцінені фільми за користувачем
user_movies = (
    positive_ratings
    .groupby("userId")["movieId"]
    .apply(lambda x: [str(movie_id) for movie_id in x])
    .tolist()
)

print("Кількість користувацьких послідовностей:", len(user_movies))

# Покажемо перші 3
for sequence in user_movies[:3]:
    print(sequence[:15])
print(len(user_movies[0]))    

Кількість користувацьких послідовностей: 609
['1', '3', '6', '47', '50', '101', '110', '151', '157', '163', '216', '231', '235', '260', '333']
['333', '1704', '3578', '6874', '46970', '48516', '58559', '60756', '68157', '74458', '79132', '80489', '80906', '86345', '89774']
['849', '1587', '2288', '2851', '3024', '3703', '4518', '5181', '5746', '5764', '5919', '6835', '7899', '7991', '26409']
200


In [5]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=user_movies,
    vector_size=50,
    window=5,
    min_count=2,
    sg=1,
    epochs=30,
    seed=42
)

print("Кількість фільмів у словнику моделі:", len(model.wv))

Кількість фільмів у словнику моделі: 3722


In [6]:
movie_id = "5"  

vector = model.wv[movie_id]

print("Розмір embedding:", len(vector))
print(vector[:10])

Розмір embedding: 50
[-0.599487   -1.2577528  -0.4050077   0.38065323 -0.44543937 -0.18229191
  0.1616136   0.25592205 -0.13103424  0.55830556]


vector_size=50 - це скільки чисел буде в одному embedding(один вектор)

In [7]:
id_to_title = dict(zip(movies["movieId"].astype(str), movies["title"]))
title_to_id = dict(zip(movies["title"], movies["movieId"].astype(str)))

print(id_to_title["2"])

Jumanji (1995)


In [8]:
movie_id = "2"

print("Фільм:", id_to_title[movie_id])
print()

similar_movies = model.wv.most_similar(movie_id, topn=10)

for similar_id, similarity in similar_movies:
    print(f"{id_to_title.get(similar_id, similar_id)} | {similarity:.3f}")

Фільм: Jumanji (1995)

American President, The (1995) | 0.962
GoldenEye (1995) | 0.959
Dangerous Minds (1995) | 0.958
Balto (1995) | 0.956
Sense and Sensibility (1995) | 0.954
Grumpier Old Men (1995) | 0.953
Babe (1995) | 0.950
Ace Ventura: When Nature Calls (1995) | 0.947
Dracula: Dead and Loving It (1995) | 0.944
Heat (1995) | 0.943


не дуже логічні, бо жанри фільмів різні
здивувало те що у всіх фільмах рік випуску 1995
думаю поруч поставила із за того що id цих фільмів дуже близькі, а списки відсортовані за id і вони часто потрапляють в один window 

In [9]:
liked_titles = [
    "Forrest Gump (1994)",
    "Shrek (2001)",
    "Star Wars: Episode VI - Return of the Jedi (1983)"
]

liked_ids = []

for title in liked_titles:
    movie_id = title_to_id.get(title)

    if movie_id is not None and movie_id in model.wv:
        liked_ids.append(movie_id)

print("Фільми користувача:")
for movie_id in liked_ids:
    print("-", id_to_title[movie_id])

Фільми користувача:
- Forrest Gump (1994)
- Shrek (2001)
- Star Wars: Episode VI - Return of the Jedi (1983)


In [10]:
vectors = [model.wv[movie_id] for movie_id in liked_ids]

user_vector = np.mean(vectors, axis=0)

print("Розмір user embedding:", user_vector.shape)
print(user_vector[:10])

Розмір user embedding: (50,)
[-0.32579902 -0.61242014 -0.11719441  0.3648049  -0.5164168   0.03998144
 -0.2713909   0.02512755 -0.11619681  0.0746946 ]


In [11]:
recommendations = model.wv.similar_by_vector(user_vector, topn=20)

print("Рекомендації:")
print()

shown = 0

for movie_id, similarity in recommendations:
    # Не рекомендуємо те, що вже було у профілі
    if movie_id in liked_ids:
        continue

    print(f"{id_to_title.get(movie_id, movie_id):50s} {similarity:.3f}")
    shown += 1

    if shown == 5:
        break

Рекомендації:

Princess Bride, The (1987)                         0.762
Good, the Bad and the Ugly, The (Buono, il brutto, il cattivo, Il) (1966) 0.762
Aliens (1986)                                      0.761
Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981) 0.751
Blues Brothers, The (1980)                         0.750


In [12]:
model_2 = Word2Vec(
    sentences=user_movies,
    vector_size=50,
    window=2,
    min_count=2,
    sg=1,
    epochs=30,
    seed=42
)

In [13]:
movie_id = title_to_id["Shrek (2001)"]

print("window=5")
for similar_id, similarity in model.wv.most_similar(movie_id, topn=5):
    print(f"{id_to_title.get(similar_id, similar_id)} | {similarity:.3f}")

print()
print("window=2")
for similar_id, similarity in model_2.wv.most_similar(movie_id, topn=5):
    print(f"{id_to_title.get(similar_id, similar_id)} | {similarity:.3f}")

window=5
Mummy Returns, The (2001) | 0.803
Knight's Tale, A (2001) | 0.798
Bridget Jones's Diary (2001) | 0.788
Eddie Murphy Raw (1987) | 0.779
Moulin Rouge (2001) | 0.776

window=2
Knight's Tale, A (2001) | 0.945
Moulin Rouge (2001) | 0.934
Mummy Returns, The (2001) | 0.932
Startup.com (2001) | 0.932
Days of Wine and Roses (1962) | 0.930


рекомендації частково змінилися 2 фільми відрізняються, а Knight's Tale, A (2001), Moulin Rouge (2001), Mummy Returns, The (2001) присутні і там і там 
думаю що це window визначає скільки сусідів бачить модель при window=2 бачить тільки 2 сусіди. тому w=5 і w=2 вчаться на різних парах фільмів  